# Lecture 4.4 — Hands-On: Build a Safe Code Review Agent (Read-Only)

In this notebook we combine **everything from Section 4** into one complete, production-safe agent:

- The permission evaluation flowchart (Lecture 4.1)
- Read-only tool surface (Lecture 4.2)
- The `safety_hook` blocking system (Lecture 4.3)
- Human-in-the-loop with `AskUserQuestion` (Lecture 3.5)

The agent will ask the user two clarifying questions, then autonomously explore a codebase using `Glob`, `Grep`, and `Read`, and produce a structured code review report — **without modifying a single file**.

### Three layers of safety

| Layer | Mechanism | What it does |
|-------|-----------|------|
| 1 — Tool whitelist | `allowed_tools=["Read", "Glob", "Grep", "AskUserQuestion"]` | Only these tools are available to the agent |
| 2 — Explicit deny | `disallowed_tools=["Write", "Edit", "Bash"]` | Write, Edit, and Bash are explicitly blocked — stated intent, not just absence |
| 3 — Hook backstop | `safety_hook` from Lecture 4.3 — fires at Gate 1 before everything else | Final automated defence even if tool lists are misconfigured |

In [1]:
# Install the Claude Agent SDK

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 MB 10.7 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Model Configuration

We pin the model name in one place so it is easy to swap. For the full list of available Claude models visit:
https://platform.claude.com/docs/en/about-claude/models/overview

In [3]:
# Model configuration
# Change this to use a different Claude model
# For the latest available models visit:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## The Sample Codebase

We create a small project with **deliberately planted issues** across six files — both security vulnerabilities and code quality problems. We will not explain exactly what is in each file right now. Part of the exercise is watching the agent discover them.

```
/content/review_project/
├── README.md
├── src/
│   ├── auth.py        ← authentication module
│   ├── database.py    ← database queries
│   ├── payments.py    ← payment processing
│   ├── utils.py       ← utility functions
│   └── api.py         ← API request handler
└── tests/
    └── test_auth.py   ← auth tests
```

In [4]:
import os

os.makedirs("/content/review_project/src", exist_ok=True)
os.makedirs("/content/review_project/tests", exist_ok=True)

with open("/content/review_project/README.md", "w") as f:
    f.write("# Review Project\n")
    f.write("A demo project for the Safe Code Review Agent.\n")

with open("/content/review_project/src/auth.py", "w") as f:
    f.write("# Authentication module\n\n")
    f.write("ADMIN_PASSWORD = 'admin123'\n")
    f.write("SECRET_KEY = 'supersecret'\n\n")
    f.write("def login(u, p):\n")
    f.write("    if p == ADMIN_PASSWORD:\n")
    f.write("        return True\n")
    f.write("    return False\n\n")
    f.write("def generate_token(user):\n")
    f.write("    return f'token_{user}'\n\n")
    f.write("def validate_token(token):\n")
    f.write("    return True\n")

with open("/content/review_project/src/database.py", "w") as f:
    f.write("# Database module\n\n")
    f.write("def get_user(username):\n")
    f.write("    query = f\"SELECT * FROM users WHERE username = '{username}'\"\n")
    f.write("    return query\n\n")
    f.write("def delete_user(user_id):\n")
    f.write("    query = f\"DELETE FROM users WHERE id = {user_id}\"\n")
    f.write("    return query\n\n")
    f.write("def update_password(username, password):\n")
    f.write("    query = f\"UPDATE users SET password = '{password}'\"\n")
    f.write("    return query\n")

with open("/content/review_project/src/payments.py", "w") as f:
    f.write("# Payments module\n\n")
    f.write("STRIPE_KEY = 'sk_live_abc123xyz'\n")
    f.write("PAYPAL_SECRET = 'paypal_secret_key_here'\n\n")
    f.write("def process_payment(amount, card):\n")
    f.write("    try:\n")
    f.write("        result = charge_card(card, amount)\n")
    f.write("    except:\n")
    f.write("        pass\n")
    f.write("    return result\n\n")
    f.write("def refund(tid, amount):\n")
    f.write("    if amount > 0:\n")
    f.write("        return {'refunded': amount}\n")

with open("/content/review_project/src/utils.py", "w") as f:
    f.write("# Utility functions\n\n")
    f.write("def fn1(x, y):\n")
    f.write("    return x + y\n\n")
    f.write("def proc(d):\n")
    f.write("    return d * 2\n\n")
    f.write("def check(e):\n")
    f.write("    return True\n\n")
    f.write("def fmt(n):\n")
    f.write("    return str(n)\n")

with open("/content/review_project/src/api.py", "w") as f:
    f.write("# API module\n\n")
    f.write("import json\n\n")
    f.write("def handle_request(req):\n")
    f.write("    data = json.loads(req)\n")
    f.write("    user = data['user']\n")
    f.write("    action = data['action']\n\n")
    f.write("    if action == 'login':\n")
    f.write("        return login(user, data['password'])\n\n")
    f.write("def parse_config(path):\n")
    f.write("    f = open(path)\n")
    f.write("    return json.load(f)\n")

with open("/content/review_project/tests/test_auth.py", "w") as f:
    f.write("# Tests for auth module\n\n")
    f.write("def test_login():\n")
    f.write("    pass\n\n")
    f.write("def test_token():\n")
    f.write("    pass\n\n")
    f.write("def test_validate():\n")
    f.write("    assert validate_token('token_admin') == True\n")

print("Sample codebase created.")

Sample codebase created.


## Verifying the Codebase

Let's print the directory structure and every file so we can see what was created — including the planted issues. When the agent produces its review, you can verify the findings against what you see here.

In [5]:
import os

print("Directory structure:")
print("=" * 40)
for root, dirs, files in os.walk("/content/review_project"):
    level = root.replace("/content/review_project", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

files_to_show = [
    "/content/review_project/README.md",
    "/content/review_project/src/auth.py",
    "/content/review_project/src/database.py",
    "/content/review_project/src/payments.py",
    "/content/review_project/src/utils.py",
    "/content/review_project/src/api.py",
    "/content/review_project/tests/test_auth.py",
]

print("\nFile contents:")
print("=" * 40)
for filepath in files_to_show:
    print(f"\n--- {filepath} ---")
    with open(filepath, "r") as f:
        print(f.read())

Directory structure:
review_project/
  README.md
  src/
    database.py
    auth.py
    api.py
    payments.py
    utils.py
  tests/
    test_auth.py

File contents:

--- /content/review_project/README.md ---
# Review Project
A demo project for the Safe Code Review Agent.


--- /content/review_project/src/auth.py ---
# Authentication module

ADMIN_PASSWORD = 'admin123'
SECRET_KEY = 'supersecret'

def login(u, p):
    if p == ADMIN_PASSWORD:
        return True
    return False

def generate_token(user):
    return f'token_{user}'

def validate_token(token):
    return True


--- /content/review_project/src/database.py ---
# Database module

def get_user(username):
    query = f"SELECT * FROM users WHERE username = '{username}'"
    return query

def delete_user(user_id):
    query = f"DELETE FROM users WHERE id = {user_id}"
    return query

def update_password(username, password):
    query = f"UPDATE users SET password = '{password}'"
    return query


--- /content/review_project/sr

## Carrying Forward the Safety Hook from Lecture 4.3

This is the complete `safety_hook` we built in Lecture 4.3 — we bring it straight into this agent.

It fires at **Gate 1** of the permission flowchart — before `allowed_tools`, before `permission_mode`, before everything else. Three blocking patterns:
1. Dangerous `Bash` commands (recursive delete, sudo, remote code execution)
2. Writes to sensitive system paths (`/etc/`, `/usr/`, `~/.ssh/`)
3. Writes outside the safe project directory — redirected automatically

In [6]:
import re
from claude_agent_sdk.types import HookMatcher

DANGEROUS_BASH_PATTERNS = [
    (r"rm\s+-rf", "recursive force delete"),
    (r"sudo", "privilege escalation"),
    (r"chmod\s+777", "unsafe file permissions"),
    (r"curl.*\|.*bash|wget.*\|.*bash", "remote code execution"),
    (r">\s*/etc/", "writing to system config"),
]

SENSITIVE_PATH_PATTERNS = [
    (r"^/etc/", "system configuration directory"),
    (r"^/usr/", "system directory"),
    (r"~/.ssh/", "SSH keys directory"),
    (r"~/.bashrc|~/.bash_profile|~/.zshrc", "shell configuration"),
    (r"^/root/", "root home directory"),
]

SAFE_DIRECTORY = "/content/review_project"


async def safety_hook(input_data, tool_use_id, context):
    tool_name = input_data["tool_name"]
    tool_input = input_data.get("tool_input", {})
    hook_event = input_data["hook_event_name"]

    if tool_name == "Bash":
        command = tool_input.get("command", "")
        for pattern, reason in DANGEROUS_BASH_PATTERNS:
            if re.search(pattern, command):
                print(f"[BLOCKED] Dangerous Bash command: {reason}")
                return {
                    "hookSpecificOutput": {
                        "hookEventName": hook_event,
                        "permissionDecision": "deny",
                        "permissionDecisionReason": (
                            f"Blocked for safety: {reason}. "
                            f"Please use a safer alternative."
                        ),
                    }
                }

    if tool_name in ["Write", "Edit"]:
        file_path = tool_input.get("file_path", "")

        for pattern, reason in SENSITIVE_PATH_PATTERNS:
            if re.search(pattern, file_path):
                print(f"[BLOCKED] Write to sensitive path: {reason}")
                return {
                    "hookSpecificOutput": {
                        "hookEventName": hook_event,
                        "permissionDecision": "deny",
                        "permissionDecisionReason": (
                            f"Writing to {file_path} was blocked: {reason}."
                        ),
                    }
                }

        if not file_path.startswith(SAFE_DIRECTORY):
            safe_path = f"{SAFE_DIRECTORY}/{os.path.basename(file_path)}"
            print(f"[MODIFIED] Redirected write: {file_path} → {safe_path}")
            return {
                "hookSpecificOutput": {
                    "hookEventName": hook_event,
                    "permissionDecision": "allow",
                    "updatedInput": {
                        **tool_input,
                        "file_path": safe_path,
                    },
                }
            }

    return {}


print("safety_hook loaded.")

safety_hook loaded.


## The AskUserQuestion Handler — Reused from Lecture 3.5

This is the same `AskUserQuestion` handler we built in Lecture 3.5 — same streaming input pattern, same dummy hook, same `can_use_tool` callback.

Key architecture point: `can_use_tool` handles `AskUserQuestion` only — it fires at **Gate 5**. The `safety_hook` handles blocking — it fires at **Gate 1**. They are registered separately and work completely independently. Hooks always fire first.

In [7]:
from claude_agent_sdk.types import PermissionResultAllow


def parse_response(response: str, options: list) -> str:
    try:
        indices = [int(s.strip()) - 1 for s in response.split(",")]
        labels = [options[i]["label"] for i in indices if 0 <= i < len(options)]
        return ", ".join(labels) if labels else response
    except ValueError:
        return response


async def handle_ask_user_question(input_data: dict):
    answers = {}

    for q in input_data.get("questions", []):
        print(f"\n{q['header']}: {q['question']}")

        options = q["options"]
        for i, opt in enumerate(options):
            print(f"  {i + 1}. {opt['label']} - {opt['description']}")
        if q.get("multiSelect"):
            print("  (Enter numbers separated by commas, or type your own answer)")
        else:
            print("  (Enter a number, or type your own answer)")

        response = input("Your choice: ").strip()
        answers[q["question"]] = parse_response(response, options)

    return PermissionResultAllow(
        updated_input={
            "questions": input_data.get("questions", []),
            "answers": answers,
        }
    )


async def can_use_tool(tool_name: str, input_data: dict, context):
    if tool_name == "AskUserQuestion":
        return await handle_ask_user_question(input_data)
    return PermissionResultAllow(updated_input=input_data)


async def dummy_hook(input_data, tool_use_id, context):
    return {"continue_": True}


print("AskUserQuestion handler loaded.")

AskUserQuestion handler loaded.


## Planning the Safe Code Review Agent

Before writing any agent code, let's be explicit about the design.

### What the agent does
1. **Ask** — Two clarifying questions via `AskUserQuestion`: review focus, report format
2. **Map** — `Glob` to discover all Python files in the project
3. **Search** — `Grep` to find common issue patterns (hardcoded strings, SQL patterns, bare excepts)
4. **Examine** — `Read` to inspect each file in detail
5. **Report** — Structured review tailored to user preferences

### The three-layer safety stack
- **Layer 1:** `allowed_tools=["Read", "Glob", "Grep", "AskUserQuestion"]` — only these tools are available
- **Layer 2:** `disallowed_tools=["Write", "Edit", "Bash"]` — write tools explicitly denied; both lists operate independently
- **Layer 3:** `safety_hook` as PreToolUse hook — fires at Gate 1, before everything else

### Why both allowed_tools and disallowed_tools?
`allowed_tools` says *only these*. `disallowed_tools` says *definitely not these*. Having both means even if `Write` were accidentally added to `allowed_tools` in a future edit, `disallowed_tools` catches it. The two lists operate independently — both have to pass.

### Hook architecture
- `dummy_hook` and `safety_hook` are **both registered as parallel `PreToolUse` hooks**
- When multiple hooks run in parallel, **deny wins over everything else**
- `can_use_tool` runs separately at Gate 5 — handles `AskUserQuestion` only

In [8]:
async def prompt_stream():
    yield {
        "type": "user",
        "message": {
            "role": "user",
            "content": """
            You are a Safe Code Review Agent. Your job is to review
            the codebase at /content/review_project for code quality
            and security issues.

            Before starting, ask the user two clarifying questions:
            You MUST use the AskUserQuestion tool to ask me clarifying questions.
            Do NOT write questions as plain text — only use the AskUserQuestion tool.

            Question 1 — Review focus:
            - Comprehensive: review both code quality and security
            - Security only: focus on vulnerabilities and security issues
            - Quality only: focus on readability, naming, and structure

            Question 2 — Report format:
            - Summary: one paragraph per file with the key issues
            - Detailed: full breakdown of every issue with suggested fixes

            Then:
            1. Use Glob to find all Python files in the project
            2. Use Grep to search for common issue patterns:
               - Hardcoded credentials and API keys
               - SQL injection patterns (f-string queries)
               - Bare except clauses
               - Poor function naming (single letter names)
               - Missing error handling
            3. Use Read to examine each file in detail
            4. Produce a structured report based on the user's preferences

            Important: Do NOT modify any files. This is a read-only review.
            """,
        },
    }

## Running the Agent

When you run the cell below, the agent will **pause twice** to ask you questions before it starts reviewing. At each question:
- Type a number and press **Enter** to choose one of the listed options
- Or type your own answer and press **Enter**

After both answers, the agent will autonomously explore the codebase and produce the report.

Try **Comprehensive + Detailed** first to see the full set of issues.

In [9]:
from claude_agent_sdk import ClaudeAgentOptions, ResultMessage, query

async for message in query(
    prompt=prompt_stream(),
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Glob", "Grep", "AskUserQuestion"],
        disallowed_tools=["Write", "Edit", "Bash"],
        can_use_tool=can_use_tool,
        model=MODEL_NAME,
        hooks={
            "PreToolUse": [
                HookMatcher(matcher=None, hooks=[dummy_hook]),
                HookMatcher(matcher=None, hooks=[safety_hook]),
            ]
        },
    ),
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)


Review Focus: What should be the focus of this code review?
  1. Comprehensive (Recommended) - Review both code quality (readability, naming, structure) and security issues (vulnerabilities, hardcoded secrets, injection risks)
  2. Security only - Focus exclusively on vulnerabilities, hardcoded credentials, injection patterns, and other security risks
  3. Quality only - Focus on readability, naming conventions, code structure, maintainability, and best practices
  (Enter a number, or type your own answer)
Your choice: 1

Report Format: How detailed should the report be?
  1. Summary (Recommended) - One paragraph per file highlighting the key issues found
  2. Detailed - Full breakdown of every issue with line numbers, code snippets, and suggested fixes
  (Enter a number, or type your own answer)
Your choice: 2
Perfect! I now have a complete picture. Let me create a comprehensive, detailed report:

---

# 🔍 COMPREHENSIVE CODE REVIEW REPORT
**Project:** /content/review_project  
**Revi

## Verifying No Files Were Modified

The agent just reviewed this codebase autonomously — finding security vulnerabilities and quality issues across multiple files. Did it modify anything?

Let's check `auth.py` — the file with the most obvious issues, including a hardcoded `ADMIN_PASSWORD`.

In [10]:
print("Verifying files were not modified by the agent:")
print("=" * 50)

with open("/content/review_project/src/auth.py", "r") as f:
    content = f.read()
    print(content)
    print()
    if "ADMIN_PASSWORD = 'admin123'" in content:
        print("\u2713 File unchanged \u2014 hardcoded password still present (agent read-only)")
    else:
        print("\u2717 File was modified \u2014 this should not happen")

Verifying files were not modified by the agent:
# Authentication module

ADMIN_PASSWORD = 'admin123'
SECRET_KEY = 'supersecret'

def login(u, p):
    if p == ADMIN_PASSWORD:
        return True
    return False

def generate_token(user):
    return f'token_{user}'

def validate_token(token):
    return True


✓ File unchanged — hardcoded password still present (agent read-only)


## Section 4 — Complete

Across four lectures you have built a complete safety toolkit for production agents:

| Lecture | What You Learned |
|---------|------------------|
| **4.1** | The 6-gate permission evaluation flowchart — how every tool request is evaluated |
| **4.2** | Read-Only vs Auto-Edit agents — the two extremes and multiple valid configurations |
| **4.3** | Hook-based blocking — automated, surgical safety rules using `PreToolUse` hooks |
| **4.4** | A complete production-safe agent with three stacked, independent layers of protection |

You now know how to build agents that are not just capable — but **safe enough for production**.

---

**Coming up — Section 5: Hooks: Controlling Agent Behavior**

In Section 5 we go deeper into hooks — using them not just for blocking, but for controlling, logging, and extending agent behaviour in powerful new ways: audit trails, notifications, dynamic behaviour changes, and more.